# FarmFederate — Weather-Modality Training (Colab GPU)

Trains the RoBERTa + ViT crop-stress model with the **AMFU Kharagpur (IMD 42893)** weather modality,
and runs the `full` / `text-only` / `none` ablation.

**Before running:** `Runtime → Change runtime type → GPU` (T4 is enough), then `Runtime → Run all`.


## 1. Check GPU


In [1]:
import torch, subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)
assert torch.cuda.is_available(), 'No GPU — set Runtime type to GPU and Run all again.'
print('torch', torch.__version__, '| CUDA device:', torch.cuda.get_device_name(0))


name, memory.total [MiB]
Tesla T4, 15360 MiB

torch 2.11.0+cu128 | CUDA device: Tesla T4


## 2. Get the code (clones `feature/multimodal-work`)
If the repo is private, you'll be prompted for a GitHub token (a fine-grained PAT with read access).


In [2]:
import subprocess, getpass, os
REPO   = 'ayushdebnath012/FarmFederate-Advisor'
BRANCH = 'feature/multimodal-work'
if os.path.isdir('repo'):
    subprocess.run(['rm','-rf','repo'])
def clone(url):
    return subprocess.run(['git','clone','--depth','1','--branch',BRANCH,url,'repo'],capture_output=True,text=True)
r = clone(f'https://github.com/{REPO}.git')
if r.returncode != 0:
    print('Public clone failed (repo is likely private).')
    tok = getpass.getpass('GitHub token: ')
    r = clone(f'https://{tok}@github.com/{REPO}.git')
assert r.returncode == 0, r.stderr
print('cloned; HEAD:')
print(subprocess.run(['git','-C','repo','log','--oneline','-1'],capture_output=True,text=True).stdout)


cloned; HEAD:
152084b Add Colab GPU notebook for weather-modality training



## 3. Install dependencies
Colab already ships torch, numpy, pandas, scikit-learn and Pillow; we add the NLP/vision + Excel bits.


In [3]:
!pip -q install 'transformers>=4.30' 'datasets>=2.12' tokenizers openpyxl 2>/dev/null
print('deps ready')


deps ready


## 4. Sanity-check the weather parser
Confirms the workbook loads and the agromet rule-labels have healthy support.


In [4]:
%cd /content/repo/backend
!python weather_data.py | tail -4


/content/repo/backend
rule-label support (days): {'water_stress': 69, 'nutrient_def': 81, 'pest_risk': 182, 'disease_risk': 225, 'heat_stress': 103}
SENSORS: soil_moisture=18.1%, soil_pH=6.2, temp=35.7°C, humidity=70%, VPD=1.5 kPa, rainfall_24h=0.0mm (trend: ↑).
WEATHER (AMFU Kharagpur/IMD 42893, 2025-05-08): Tmax=38.0°C, Tmin=25.0°C, RH ~76% morning / ~61% evening, rain_24h=0.0mm, rain_7d=25.0mm, rainless_days=1, dew_point=~24.9°C, VPD=1.46kPa, wind=SW 4kmph, pan_evap=~4.4mm, soil_temp_5cm=31.0°C, sunshine=7.8h.
labels: ['heat_stress'] features: [1.2000000476837158, 0.6000000238418579, 0.8999999761581421, -0.33000001311302185, 0.07000000029802322, -0.33000001311302185, -0.10999999940395355, -0.23000000417232513, -0.6000000238418579, 0.3199999928474426, 0.8199999928474426, -0.33000001311302185, 0.1599999964237213, 0.6700000166893005, 0.9300000071525574, 1.5, -0.9900000095367432, 1.149999976158142, -0.8399999737739563]


## 5. (optional) Mount Drive to keep checkpoints
Skip this cell to keep outputs only in the Colab session.


In [5]:
USE_DRIVE = False  # set True to persist to Drive
OUT = '/content/checkpoints'
if USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/FarmFederate/checkpoints'
import os; os.makedirs(OUT, exist_ok=True); print('checkpoints ->', OUT)


checkpoints -> /content/checkpoints


## 6. Train — full weather modality
Unfrozen backbones, real plant-stress images (auto-downloaded), mixed precision.
Adjust `--epochs`, `--max-samples`, `--max-images` to trade speed for quality.


In [ ]:
!python -u multimodal_train.py \
    --weather-mode full \
    --epochs 6 --batch-size 32 --lr 3e-5 --amp \
    --max-samples 6000 --max-per-source 2000 \
    --max-images 4000 --max-per-image-dataset 2000 \
    --out $OUT --tag colab


[run] central_full_colab → /content/checkpoints/central_full_colab (device=cuda)
[Weather] 974 observed days loaded (mode=full); rule-label support: {'water_stress': 69, 'nutrient_def': 81, 'pest_risk': 182, 'disease_risk': 225, 'heat_stress': 103}
[Mix] loading gardian (<= 2000) ...
README.md: 100% 5.01k/5.01k [00:00<00:00, 12.4MB/s]
[WARN] CGIAR/gardian-ai-ready-docs failed: Dataset 'CGIAR/gardian-ai-ready-docs' is a gated dataset on the Hub. You must be authenticated to access it., trying next...
[WARN] maharshipandya/agricultural-datasets failed: Dataset 'maharshipandya/agricultural-datasets' doesn't exist on the Hub or cannot be accessed., trying next...
[WARN] turing-motors/agricultural-qa failed: Dataset 'turing-motors/agricultural-qa' doesn't exist on the Hub or cannot be accessed., trying next...
[Mix] gardian added 0 rows
[Mix] loading argilla (<= 2000) ...
README.md: 100% 15.2k/15.2k [00:00<00:00, 31.7MB/s]

data/train-00000-of-00001.parquet: downloading bytes:  47% 1.19M/2.

## 7. Ablation — `text-only` and `none`
Same recipe, different use of the station data, for the weather-vs-no-weather comparison.
Comment this out if you only need the `full` model.


In [ ]:
for MODE in ['text-only','none']:
    print('='*30, MODE, '='*30)
    !python -u multimodal_train.py --weather-mode {MODE} --epochs 6 --batch-size 32 --lr 3e-5 --amp \
        --max-samples 6000 --max-per-source 2000 --max-images 4000 --max-per-image-dataset 2000 --out $OUT --tag colab


## 8. Compare the runs


In [ ]:
import json, glob, pandas as pd
rows = []
for f in sorted(glob.glob(f'{OUT}/central_*_colab/metrics.json')):
    d = json.load(open(f)); v = d['best_val']
    rows.append({'mode': d['weather_mode'], 'best_epoch': d['best_epoch'],
                 'f1_macro': round(v['f1_macro'],4), 'f1_micro': round(v['f1_micro'],4),
                 **{k: round(x,3) for k,x in v['f1_per_label'].items()}})
pd.DataFrame(rows).set_index('mode') if rows else print('no metrics yet')


## 9. (optional) Download the best `full` checkpoint


In [ ]:
from google.colab import files
ckpt = f'{OUT}/central_full_colab/global_central.pt'
import os; files.download(ckpt) if os.path.exists(ckpt) else print('checkpoint not found:', ckpt)
